# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Descrição
Este notebook tem como objetivo analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Etapas de Pré-processamento
Este notebook descreve as etapas de pré-processamento necessárias para analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Necessidades de Processamento
- **Identificar o ponto de divisão** entre as reflexões do Terço e as reflexões do Dia.
- **Separar os textos** em reflexões do Terço e reflexões do Dia.
- **Remover as seções** marcadas com a tag `[Música]`.
- **Identificar e remover orações oficiais** da Igreja Católica.
- **Extrair e documentar músicas cantadas**, incluindo o nome e o autor de cada música.
- **Considerar o momento do Rosário** em que cada ensinamento foi apresentado, relacionando-o ao terço e ao mistério meditado naquele momento.

## 1.0. Importação de Bibliotecas


In [ ]:
from langchain.prompts import PromptTemplate
from langchain_ollama import ChatOllama

In [ ]:
# MODEL = "deepseek-r1:8b"
MODEL = "gemma3:12b"

# Initialize the model
llm = ChatOllama(model=MODEL, temperature=0.1)

## 2.0. Processamento de texto

In [ ]:
import re
from typing import List, TypedDict

from pydantic import BaseModel, Field
from sympy import div


class FullTextDivisions(BaseModel):
    rosario: List[str] = Field(
        description="Texto do Santo Rosário",
    )
    reflexao: List[str] = Field(
        description="Texto da Reflexão do Dia",
    )


class State(TypedDict):
    full_text: str
    rosario: str
    reflexao: str
    musicas: List[str]


def clean_text(state: State) -> str:
    """Função para limpar texto

    Retira a anotação "[Música]"
    Retira marcas temporais com o seguinte formato: "(22:31)" e "(122:31)"

    Args:
        input_text (_type_): Texto a ser limpo
    """

    # Remove "[Música]"
    cleaned_text = state["full_text"].replace("[Música]", "")

    # Remove timestamps in the format "(22:31)" e "(122:31)"
    cleaned_text = re.sub(r"\(\d{1,3}:\d{2}\)", "", cleaned_text)

    return {"full_text": cleaned_text}


def fix_text(state: State):
    """Função para corrigir texto

    Args:
        input_text (_type_): Texto a ser corrigido

    Returns:
        _type_: _description_
    """
    fix_text_prompt_template = PromptTemplate.from_template(
        """
        Você é especialista em limpar textos quebrados e mal formatados,
        por exemplo: quebras de linha em lugares estranhos etc. 
        Além disso, você tem a capacidade de adicionar a pontuação adequada para melhorar a legibilidade e a clareza. 
        Etapas:
            - Leia o documento inteiro e compreenda-o completamente.
            - Remova todas as quebras de linha estranhas que atrapalhem a formatação.
            - NÃO altere nenhum conteúdo ou ortografia.
            - Analise o texto e identifique as áreas em que a pontuação está ausente ou incorreta.
            - Insira os sinais de pontuação adequados (vírgulas, pontos, pontos de interrogação etc.)
            para melhorar a estrutura e o fluxo das frases.
            
        INSTRUÇÕES DE PRODUÇÃO:
            - Produza o texto completo, devidamente formatado e com a pontuação correta.
            - Não produza avisos ou notas
            - apenas as seções solicitadas.
            
        Você vai retornar apenas o texto reformatado! Sem qualquer introdução ou explicação.
            
        Texto a ser corrigido:
        
        {input_text}
        """
    )
    
    # Generate the prompt
    prompt = fix_text_prompt_template.format(input_text=state["full_text"])

    # Get the response from the model
    response = llm.invoke(prompt)

    # Return the cleaned text
    return {"full_text": response.content}


def remove_pray(state: State):
    """Função para remover a oração do texto

    Args:
        input_text (_type_): Texto a ser removido

    Returns:
        _type_: _description_
    """
    prompt_template = PromptTemplate.from_template(
        """
        O texto contém orações católicas amplamente conhecidas pelos fieis,
        como o Pai Nosso, Ave Maria, Credo, Vinde Espírito Santo, Salve Rainha, entre outras.
        
        Você precisa remover completamente os trechos que contêm essas orações.
        Você não pode alterar o conteúdo restante ou a formatação do texto.
        
        Texto a ser removido:
        
        {input_text}
        """
    )

    # Generate the prompt
    prompt = prompt_template.format(input_text=state["full_text"])

    # Get the response from the model
    response = llm.invoke(prompt)

    # Return the cleaned text
    return {"full_text": response.content}


def slit_text(state: State):
    """Função para dividir o texto de entrada em relação ao Rosário e à reflexão do dia
    utilizando LLMs

    Args:
        input_text (_type_): Texto a ser dividido

    Returns:
        _type_: _description_
    """

    prompt_template = PromptTemplate.from_template(
        """
        O texto contém duas partes bem distintas:
        
        1. O Santo Rosário, que também conta com diversos ensinamentos e reflexões religiosas católicas.
        2. A reflexão do dia, que acontece sempre após a finalização do santo rosário.
        
        Você precisa separar as partes e retornar ambas.
        
        Você não pode alterar nenhuma palavra, nem pontuação nem ordem alguma entre quaisquer palavras dentro da divisão.
        
        Texto a ser dividido:
        
        {input_text}
        """
    )
    structured_llm = llm.with_structured_output(FullTextDivisions)

    # Generate the prompt
    prompt = prompt_template.format(input_text=state["full_text"])
    
    # Get the response from the model
    divisions = structured_llm.invoke(prompt)

    # Return the two parts
    return {"rosario": divisions.rosario, "reflexao": divisions.reflexao}


def get_musics(state: State):
    """Função para extrair as músicas do texto

    Args:
        input_text (_type_): Texto a ser extraído

    Returns:
        _type_: _description_
    """

    prompt_template = PromptTemplate.from_template(
        """
        O texto contém várias músicas católicas, que não estão necessariamente separadas por qualquer marca no texto.
        
        Elas são anunciadas ao longo do texto, mesmo sem haver separação.
        
        Você precisa extrair todas as músicas e retornar uma lista com os nomes delas e autores.
        
        Cada música em uma linha, separada do autor por um hífen.
        
        Como no exemplo: "Música - Autor"
        
        Texto a ser extraído:
        
        {input_text}
        """
    )

    # Generate the prompt
    prompt = prompt_template.format(input_text=state.full_text)

    # Get the response from the model
    response = llm.invoke(prompt)

    # Return the cleaned text
    return {"musicas": response.content.split("\n")}

In [ ]:
from re import split
from langgraph.graph import START, StateGraph
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.prompts import PromptTemplate

graph_builder = StateGraph(State).add_sequence([clean_text, fix_text, slit_text, remove_pray, get_musics])
graph_builder.add_edge(START, "clean_text")
graph = graph_builder.compile()

## 3.0. Leitura dos Arquivos

In [ ]:
def proccess_text(state: State) -> State:
    """Função para processar o texto

    Args:
        state (State): Estado atual do texto

    Returns:
        State: Estado atualizado do texto
    """

    # Process the text using the graph
    # Stream steps (demonstration porpose only)
    for step in graph.stream(
        {"full_text": state["full_text"]},
        stream_mode="updates",
    ):
        print(f"{step}\n\n----------------\n")

    # Return the updated state
    return state

In [ ]:
import os

from tqdm import tqdm

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo(ordem):
    return titulo_template.format(ordem=ordem)


for i in tqdm(range(1, 41), desc="Processando arquivos", unit="arquivo"):
    titulo = gerar_titulo(str(i))
    arquivo = f"{raw_folder}/{titulo}"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                print(f"\n\nArquivo: {titulo}")
                print(f"\nPrévia do conteúdo do arquivo:\n")
                print(conteudo[:500])  # Exibe os primeiros 500 caracteres do conteúdo
                print("\n\n")

                proccess_text_state = {
                    "full_text": conteudo,
                    "rosario": "",
                    "reflexao": "",
                    "musicas": "",
                }
                # Process the text
                state = proccess_text(proccess_text_state)
                # Print the results
                print("\n\nTexto limpo:\n")

                # Write a file for each state
                with open(f"{processed_folder}/{titulo}", "w+", encoding="utf-8") as f:
                    f.write(state["full_text"])

                with open(
                    f"{processed_folder}/rosario_{titulo}", "w+", encoding="utf-8"
                ) as f:
                    f.write(state["rosario"])

                with open(
                    f"{processed_folder}/reflexao_{titulo}", "w+", encoding="utf-8"
                ) as f:
                    f.write(state["reflexao"])

                with open(
                    f"{processed_folder}/musicas_{titulo}", "w+", encoding="utf-8"
                ) as f:
                    f.write("\n".join(state["musicas"]))

            else:
                print(f"\n\nO arquivo {titulo} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")

    break